In [1]:
# pip install jupyter jupyterlab jupyterlab_h5web
import h5py
import h5py.h5p as h5p
import numpy as np
from jupyterlab_h5web import H5Web

prefix = "mem_opt_nomad_track"

In [2]:
for compress in [True, False]:
    for storage_layout in ["contiguous", "chunked"]:
        if storage_layout == "contiguous":
            suffix = ""
            if not compress:
                continue
        else:
            suffix = f"_{'compressed' if compress else 'uncompressed'}"

        prng = np.random.default_rng(seed=42)  # deterministic seeding
        n = 50 * 2_500_000  # _000  # float32, 10 GB make reasonably large to see effect on RAM consumption but not too large to have test system crashing
        if storage_layout == "chunked":
            suffix = f"_{'compressed' if compress else 'uncompressed'}"
        else:
            suffix = ""
        
        with h5py.File(f"{prefix}_{n}_{storage_layout}{suffix}.nxs", "w", track_order=True) as h5w:
            gcpl = h5w.id.get_create_plist()
            flags = gcpl.get_link_creation_order()
        
            tracked = bool(flags & h5p.CRT_ORDER_TRACKED)
            indexed = bool(flags & h5p.CRT_ORDER_INDEXED)
        
            print("track_order (tracked):", tracked)
            print("track_order (indexed):", indexed)

            axes = ["indices_image", "axis_j", "axis_i"]
            intensity = np.asarray(prng.random(size=n), np.float32).reshape(-1, 50, 50)  # [0, 1), last is fastest
            shape = np.shape(intensity)
            print(f"{axes}, {shape}")
            
            grp = h5w.create_group("/entry1")
            grp.attrs["NX_class"] = "NXentry"
            dst = h5w.create_dataset("/entry1/definition", data="NXem")
            grp = h5w.create_group("/entry1/measurement")
            grp.attrs["NX_class"] = "NXem_measurement"
            grp = h5w.create_group("/entry1/measurement/event1")
            grp.attrs["NX_class"] = "NXem_event_data"
            grp = h5w.create_group("/entry1/measurement/event1/image1")
            grp.attrs["NX_class"] = "NXimage"
            grp = h5w.create_group("/entry1/measurement/event1/image1/stack_2d")
            grp.attrs["NX_class"] = "NXdata"
            trg = "/entry1/measurement/event1/image1/stack_2d"
            grp.attrs["axes"] = axes
            grp.attrs["signal"] = "real"
            for idx, axis in enumerate(axes):
                grp.attrs[f"{axis}_indices"] = np.uint32(idx)  # len(axes) - idx
            # having dst chunked should 
            if storage_layout == "chunked":  # should trigger chunk iteration-based functions for iuf, here aiming for 1 MB chunks 250_000 * 4 B
                if compress:
                    # comp = {"compression": "gzip",
                    #         "compression_opts": 1}
                    dst = h5w.create_dataset(f"{trg}/real", compression="gzip", compression_opts=1, chunks=(100, 50, 50), data=intensity)
                else:
                    dst = h5w.create_dataset(f"{trg}/real", chunks=(100, 50, 50), data=intensity)                
            else:  # classical approach contiguous storage
                dst = h5w.create_dataset(f"{trg}/real", data=intensity)
            dst.attrs["long_name"] = "real"
            for idx, axis in enumerate(axes):
                dst = h5w.create_dataset(f"{trg}/{axis}", data=np.asarray(np.arange(shape[idx]), np.uint32))
                dst.attrs["long_name"] = axis

track_order (tracked): True
track_order (indexed): True
['indices_image', 'axis_j', 'axis_i'], (50000, 50, 50)
track_order (tracked): True
track_order (indexed): True
['indices_image', 'axis_j', 'axis_i'], (50000, 50, 50)
track_order (tracked): True
track_order (indexed): True
['indices_image', 'axis_j', 'axis_i'], (50000, 50, 50)


In [ ]:
H5Web(file_path)